# 02 – Candidate ligand–receptor pairs from CD8αα(I)/NK to DC1/aDC1

**Question.** Which ligand–receptor (L–R) pairs from the authors' CellPhoneDB table (S5) are candidates for the run
**senders (CD8αα(I), NK) → receivers (DC1, aDC1)**, and where do the claimed pairs XCL1 → XCR1 and XCL2 → XCR1 stand?

The method is that of thymus case 1, notebook 03, reused without change:
1. orient every S5 pair as ligand → receptor with the CellPhoneDB v2 `receptor` / `integrin` flags, plus the same
   documented manual corrections (the order of the two names in an S5 pair says nothing about the direction);
2. keep a pair if the ligand is detected in ≥ 10% of the cells of a sender population and the receptor in ≥ 10% of a
   receiver population (for a complex: its weakest subunit);
3. compare with the S5 values, read from the correctly oriented column.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

ROOT = Path("../..")
THY = ROOT / "data/thymus"
DB = THY / "cellphonedb"
OUT = ROOT / "results/thymus_case2_xcl1/02_ligand_receptor_candidates"
OUT.mkdir(parents=True, exist_ok=True)

SENDERS = ["CD8αα(I)", "NK"]
RECEIVERS = ["DC1", "aDC1"]

s5 = pd.read_csv(THY / "aay3224_table-s5.csv", header=1, low_memory=False).drop(columns="Unnamed: 0")
prot = pd.read_csv(DB / "protein_input.csv").set_index("uniprot")
cplx = pd.read_csv(DB / "complex_input.csv").set_index("complex_name")
gene = pd.read_csv(DB / "gene_input.csv").drop_duplicates("uniprot").set_index("uniprot")
# The count matrix uses older names for the nectins (PVRL1-3); rename them to match
gene["gene_name"] = gene.gene_name.replace({"NECTIN1": "PVRL1", "NECTIN2": "PVRL2", "NECTIN3": "PVRL3"})
print("S5 pairs:", len(s5))

S5 pairs: 710


## 1. Partners → genes, receptor flags and orientation

Same rules as thymus case 1 (notebook 03): genes are matched by symbol through the UniProt ID; manual ligands first,
then the receptor flag, then the integrin flag; pairs that remain ambiguous are kept in both orientations.

In [2]:
def partner_info(p):
    kind, name = p.split(":", 1)
    if kind == "simple":
        if name not in prot.index:
            return dict(genes=None, receptor=None, integrin=None)
        return dict(genes=[gene.gene_name.get(name)], receptor=bool(prot.receptor[name]),
                    integrin=bool(prot.integrin[name]))
    row = cplx.loc[name]
    units = [u for u in row[["uniprot_1", "uniprot_2", "uniprot_3", "uniprot_4"]].dropna()]
    return dict(genes=[gene.gene_name.get(u) for u in units], receptor=bool(row.receptor), integrin=bool(row.integrin))

rows = []
for _, r in s5.iterrows():
    a, b = partner_info(r.partner_a), partner_info(r.partner_b)
    rows.append(dict(pair=r.interacting_pair, partner_a=r.partner_a, partner_b=r.partner_b,
                     genes_a=a["genes"], genes_b=b["genes"], receptor_a=a["receptor"], receptor_b=b["receptor"],
                     integrin_a=a["integrin"], integrin_b=b["integrin"]))
lr = pd.DataFrame(rows)
print("pairs with a partner missing from the database:",
      lr.loc[lr.genes_a.isna() | lr.genes_b.isna(), "pair"].tolist())
lr = lr.dropna(subset=["genes_a", "genes_b"]).reset_index(drop=True)

MANUAL_LIGANDS = {"TNFSF11", "CXCL16", "HBEGF", "NRG1", "SEMA4C", "SEMA4D"}
def orient(r):
    sa = set(r.genes_a) if r.partner_a.startswith("simple") else set()
    sb = set(r.genes_b) if r.partner_b.startswith("simple") else set()
    if sa & MANUAL_LIGANDS and not sb & MANUAL_LIGANDS: return "a", "manual"
    if sb & MANUAL_LIGANDS and not sa & MANUAL_LIGANDS: return "b", "manual"
    if r.receptor_a != r.receptor_b:                    return ("b" if r.receptor_a else "a"), "receptor flag"
    if r.integrin_a != r.integrin_b:                    return ("b" if r.integrin_a else "a"), "integrin"
    return None, "ambiguous"

lr[["ligand_side", "orientation_rule"]] = lr.apply(lambda r: pd.Series(orient(r)), axis=1)

def as_oriented(r, side):
    lig, rec = ("a", "b") if side == "a" else ("b", "a")
    return dict(pair=r.pair, ligand=r["partner_" + lig].split(":", 1)[1], receptor=r["partner_" + rec].split(":", 1)[1],
                ligand_genes=r["genes_" + lig], receptor_genes=r["genes_" + rec],
                ligand_is_partner_a=(lig == "a"), orientation_rule=r.orientation_rule)

oriented = pd.DataFrame([as_oriented(r, s) for _, r in lr.iterrows()
                         for s in ([r.ligand_side] if r.ligand_side else ["a", "b"])])
oriented["ligand"] = [g[0] if l in gene.index else l for l, g in zip(oriented.ligand, oriented.ligand_genes)]
oriented["receptor"] = [g[0] if l in gene.index else l for l, g in zip(oriented.receptor, oriented.receptor_genes)]
print(lr.orientation_rule.value_counts().to_string())
oriented[oriented.pair.isin(["XCL1_XCR1", "XCL2_XCR1", "CD74_MIF", "CCL19_CCR7"])][
    ["pair", "ligand", "receptor", "ligand_is_partner_a", "orientation_rule"]]

pairs with a partner missing from the database: ['RIF1_NOTCH1']
orientation_rule
receptor flag    431
integrin         208
ambiguous         55
manual            15


,pair,ligand,receptor,ligand_is_partner_a,orientation_rule
19,CCL19_CCR7,CCL19,CCR7,True,receptor flag
190,CD74_MIF,MIF,CD74,False,receptor flag
384,XCL1_XCR1,XCL1,XCR1,True,receptor flag
385,XCL2_XCR1,XCL2,XCR1,True,receptor flag


## 2. Expression in the selected populations

Fraction of cells with at least one raw count, per population, for every S5 gene. Cells come from notebook 01.

In [3]:
cells = pd.read_csv(ROOT / "results/thymus_case2_xcl1/01_cell_populations/selected_cells.csv", index_col=0)
raw = ad.read_h5ad(THY / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")

all_genes = sorted({g for gs in pd.concat([oriented.ligand_genes, oriented.receptor_genes]) for g in gs if g})
gene_pos = pd.Series(np.arange(raw.n_vars), index=raw.var_names)
missing = [g for g in all_genes if g not in gene_pos.index]
print("S5 genes:", len(all_genes), "| not in the count matrix:", missing)
all_genes = [g for g in all_genes if g in gene_pos.index]
assert not set(all_genes) & set(raw.var_names[raw.var_names.duplicated()])

row_pos = np.sort(raw.obs_names.get_indexer(cells.index))
X = raw.X[row_pos][:, gene_pos[all_genes].values]
pop_of_row = cells.loc[raw.obs_names[row_pos], "population"].values

detected = pd.DataFrame((X > 0).toarray(), columns=all_genes)
frac = detected.groupby(pop_of_row).mean()
frac.shape

/tmp/ipykernel_451/2274239372.py:2: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  raw = ad.read_h5ad(THY / "HTA07.A01.v02.entire_data_raw_count.h5ad", backed="r")


S5 genes: 516 | not in the count matrix: []


(4, 516)

In [4]:
THRESHOLD = 0.10

def side_fraction(genes, pops):
    genes = [g for g in genes if g in frac.columns]
    if not genes:
        return pd.Series(0.0, index=pops)
    return frac.loc[pops, genes].min(axis=1)               # complex: weakest subunit

def candidates(senders, receivers, name):
    out = []
    for _, r in oriented.iterrows():
        fl, fr = side_fraction(r.ligand_genes, senders), side_fraction(r.receptor_genes, receivers)
        if fl.max() >= THRESHOLD and fr.max() >= THRESHOLD:
            out.append(dict(direction=name, pair=r.pair, ligand=r.ligand, receptor=r.receptor,
                            orientation_rule=r.orientation_rule, ligand_is_partner_a=r.ligand_is_partner_a,
                            max_ligand_frac=fl.max(), ligand_pops=", ".join(fl.index[fl >= THRESHOLD]),
                            max_receptor_frac=fr.max(), receptor_pops=", ".join(fr.index[fr >= THRESHOLD])))
    return pd.DataFrame(out)

cand = candidates(SENDERS, RECEIVERS, "CD8aa(I)/NK -> DC1/aDC1")
print("candidates:", len(cand))
cand.groupby("orientation_rule").size()

candidates: 59


orientation_rule
ambiguous        10
integrin          3
manual            2
receptor flag    44
dtype: int64

## 3. Comparison with the S5 values

S5 column `X_Y` holds the mean of partner_a in X and partner_b in Y, so the sender → receiver value is in
`sender_receiver` when the ligand is partner_a and in `receiver_sender` otherwise. S5 labels: CD8αα(I) is `CD8aa(I)`,
and the three aDC subtypes are merged into `aDC`, so the aDC1 values are approximate.

In [5]:
TO_S5 = {"CD8αα(I)": "CD8aa(I)", "aDC1": "aDC"}
s5v = s5.set_index("interacting_pair")
for s in SENDERS:
    for t in RECEIVERS:
        S, T = TO_S5.get(s, s), TO_S5.get(t, t)
        assert f"{S}_{T}" in s5v.columns and f"{T}_{S}" in s5v.columns

def s5_best(r, senders, receivers):
    best = (0.0, "")
    for s in senders:
        for t in receivers:
            S, T = TO_S5.get(s, s), TO_S5.get(t, t)
            col = f"{S}_{T}" if r.ligand_is_partner_a else f"{T}_{S}"
            v = float(s5v.loc[r.pair, col])
            if v > best[0]:
                best = (v, f"{s} -> {t}")
    return pd.Series(best, index=["s5_max_mean", "s5_best_pair"])

cand[["s5_max_mean", "s5_best_pair"]] = cand.apply(s5_best, axis=1, senders=SENDERS, receivers=RECEIVERS)

show = ["pair", "ligand", "receptor", "orientation_rule", "max_ligand_frac", "ligand_pops",
        "max_receptor_frac", "receptor_pops", "s5_max_mean", "s5_best_pair"]
cand.sort_values("s5_max_mean", ascending=False)[show].head(35).round(2)

,pair,ligand,receptor,orientation_rule,max_ligand_frac,ligand_pops,max_receptor_frac,receptor_pops,s5_max_mean,s5_best_pair
25,CD74_MIF,MIF,CD74,receptor flag,0.72,"CD8αα(I), NK",1.00,"DC1, aDC1",3.86,CD8αα(I) -> DC1
7,APP_CD74,APP,CD74,receptor flag,0.22,"CD8αα(I), NK",1.00,"DC1, aDC1",2.90,CD8αα(I) -> DC1
18,MIF_TNFRSF14,MIF,TNFRSF14,receptor flag,0.72,"CD8αα(I), NK",0.59,"DC1, aDC1",1.42,CD8αα(I) -> aDC1
48,XCL1_XCR1,XCL1,XCR1,receptor flag,0.67,"CD8αα(I), NK",0.56,DC1,1.12,CD8αα(I) -> DC1
5,LGALS9_CD44,LGALS9,CD44,receptor flag,0.12,CD8αα(I),0.80,"DC1, aDC1",1.11,NK -> aDC1
49,XCL2_XCR1,XCL2,XCR1,receptor flag,0.84,NK,0.56,DC1,0.91,NK -> DC1
47,CLEC2B_KLRF1,KLRF1,CLEC2B,ambiguous,0.46,NK,0.26,DC1,0.86,NK -> DC1
40,CD28_CD86,CD28,CD86,receptor flag,0.54,CD8αα(I),0.63,"DC1, aDC1",0.84,CD8αα(I) -> aDC1
11,CD40LG_CD40,CD40LG,CD40,receptor flag,0.11,CD8αα(I),0.90,"DC1, aDC1",0.81,CD8αα(I) -> aDC1
1,CCL3_IDE,CCL3,IDE,receptor flag,0.44,NK,0.10,DC1,0.80,NK -> DC1


**Reading.**
- 59 oriented pairs pass the 10% filter (receptor flag 44, ambiguous 10, integrin 3, manual 2), fewer than the 132 of
  thymus case 1 because there are only two populations on each side.
- As in case 1, the top of the S5 ranking is held by ubiquitous genes (MIF, APP → CD74). The claimed pairs come right
  after: **XCL1 → XCR1 is fourth (1.12) and XCL2 → XCR1 sixth (0.91)**, so the authors' own table ranks them among the
  strongest signals between these populations.
- Most other candidates are T/NK–DC contact and costimulation pairs (CD28 → CD86/CD80, CD40LG → CD40, LTB → LTBR,
  PDCD1–CD274, ICAM1 and its integrins, FLT3LG → FLT3) and IFNG from NK cells to the IFN-γ receptor on DCs.
- `CCL3_IDE` has an enzyme as "receptor"; it is excluded in notebook 03, as in case 1.

## 4. The claimed pairs

In [6]:
CLAIMS = {"XCL1_XCR1": "XCL1 from CD8αα(I) (and NK) -> XCR1 on DC1",
          "XCL2_XCR1": "XCL2, paralogue of XCL1, same receptor"}
claim_rows = []
for pair, text in CLAIMS.items():
    o = oriented[oriented.pair == pair].iloc[0]
    hit = cand[cand.pair == pair]
    S5 = {f"{s} -> {t}": float(s5v.loc[pair, f"{TO_S5.get(s, s)}_{TO_S5.get(t, t)}" if o.ligand_is_partner_a
                                       else f"{TO_S5.get(t, t)}_{TO_S5.get(s, s)}"])
          for s in SENDERS for t in RECEIVERS}
    claim_rows.append(dict(pair=pair, claim=text, oriented_as=f"{o.ligand} -> {o.receptor}",
                           orientation_rule=o.orientation_rule, candidate=len(hit) > 0, **S5))
pd.DataFrame(claim_rows).round(2)

,pair,claim,oriented_as,orientation_rule,candidate,CD8αα(I) -> DC1,CD8αα(I) -> aDC1,NK -> DC1,NK -> aDC1
0,XCL1_XCR1,XCL1 from CD8αα(I) (and NK) -> XCR1 on DC1,XCL1 -> XCR1,receptor flag,True,1.12,0.83,0.90,0.61
1,XCL2_XCR1,"XCL2, paralogue of XCL1, same receptor",XCL2 -> XCR1,receptor flag,True,0.81,0.52,0.91,0.62


In [7]:
# Detection fractions behind the claim and the activation read-out, in the four populations
claim_genes = ["XCL1", "XCL2", "XCR1", "CCR7", "CCL19", "CCL21", "CD40LG", "CD40"]
frac.loc[SENDERS + RECEIVERS, [g for g in claim_genes if g in frac.columns]].round(2)

,XCL1,XCL2,XCR1,CCR7,CCL19,CCL21,CD40LG,CD40
CD8αα(I),0.18,0.09,0.00,0.13,0.01,0.0,0.11,0.00
NK,0.67,0.84,0.00,0.15,0.02,0.0,0.01,0.01
DC1,0.01,0.00,0.56,0.04,0.07,0.0,0.01,0.43
aDC1,0.04,0.01,0.02,0.90,0.68,0.0,0.03,0.90


**Reading.**
- Both claimed pairs are oriented XCL → XCR1 by the receptor flag (no manual correction) and pass the filter: XCL1 in
  CD8αα(I) (18%) and NK (67%), XCL2 in NK only (84%), XCR1 in DC1 (56%) but not in aDC1 (2%).
- In S5, XCL1 → XCR1 is higher from CD8αα(I) (1.12) than from NK (0.90) to DC1, as the paper says. In the public
  object, NK is still first on the linear scale that CellPhoneDB uses (notebook 01: 97.8 against 78.5 per 100,000).
  S5 was computed on an earlier annotation, so the gap cannot be traced further.
- CCR7 is in 90% of aDC1 cells, but its ligands are not in the senders (CCL19 in 1–2%, CCL21 absent; CCL19 is made by
  aDC1 itself). CCR7 is therefore not an L–R link of this run; it enters notebook 04 as an activation read-out on the
  receiver side.
- CD40LG → CD40, the classical T-cell signal that activates DCs, passes thanks to CD8αα(I) (11%).

In [8]:
cand.to_csv(OUT / "candidates_cd8aa_nk_to_dc1.csv", index=False)
oriented.drop(columns=["ligand_genes", "receptor_genes"]).to_csv(OUT / "s5_oriented.csv", index=False)
frac.T.round(3).to_csv(OUT / "expression_fraction_by_population.csv")
print("written to", OUT)

written to ../../results/thymus_case2_xcl1/02_ligand_receptor_candidates


## Summary

| | |
|---|---|
| S5 pairs | 710; 1 dropped (`RIF1_NOTCH1`, partner not in the database) |
| Orientation | same rules and same counts as thymus case 1 (receptor flag 431 · integrin 208 · manual 15 · ambiguous 55) |
| Candidates, CD8αα(I)/NK → DC1/aDC1 | 59 oriented pairs pass the 10% filter |
| Claimed pairs | XCL1 → XCR1 and XCL2 → XCR1 both pass; 4th and 6th by S5 mean |

Next (notebook 03): the CausalCCC input – cell cap, the 25 L–R pairs (claimed pairs forced in, the rest by S5 mean).